# 桥梁涡激振动风险评估 - 快速开始 🚀

这个notebook帮助你快速了解和运行桥梁VIV风险评估系统。

## 1. 环境设置

In [ ]:
import sys
import os
from pathlib import Path

# 添加src路径
project_root = Path('.').resolve().parent
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源代码目录: {src_path}")

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表样式
sns.set_style("whitegrid")
plt.style.use('default')

print("📦 库导入完成!")

## 2. 数据加载和探索

In [ ]:
# 加载数据
data_path = project_root.parent / 'bridge_dataset_fixed.csv'
print(f"数据文件路径: {data_path}")

if data_path.exists():
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    print(f"✅ 数据加载成功!")
    print(f"📊 数据形状: {df.shape}")
    print(f"📋 列名: {list(df.columns)}")
else:
    print(f"❌ 数据文件不存在: {data_path}")

In [ ]:
# 数据基本信息
print("📈 数据基本信息:")
print(df.info())

print("\n📊 数值列统计:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(df[numeric_cols].describe())

In [ ]:
# 查看前几行数据
print("🔍 前5行数据:")
df.head()

## 3. 数据可视化

In [ ]:
# 风险等级分布
if 'Risk_Level' in df.columns:
    plt.figure(figsize=(10, 6))
    
    plt.subplot(1, 2, 1)
    risk_counts = df['Risk_Level'].value_counts()
    plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', startangle=90)
    plt.title('风险等级分布')
    
    plt.subplot(1, 2, 2)
    sns.countplot(data=df, x='Risk_Level', order=['Low', 'Medium', 'High'])
    plt.title('风险等级计数')
    plt.xlabel('风险等级')
    plt.ylabel('桥梁数量')
    
    plt.tight_layout()
    plt.show()
    
    print("📊 风险等级统计:")
    for level, count in risk_counts.items():
        percentage = count / len(df) * 100
        print(f"  {level}: {count}座桥梁 ({percentage:.1f}%)")

In [ ]:
# 振幅分布
if 'Max_Amplitude_mm' in df.columns:
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.hist(df['Max_Amplitude_mm'], bins=15, alpha=0.7, color='skyblue')
    plt.xlabel('最大振幅 (mm)')
    plt.ylabel('频数')
    plt.title('振幅分布直方图')
    
    plt.subplot(1, 3, 2)
    plt.boxplot(df['Max_Amplitude_mm'])
    plt.ylabel('最大振幅 (mm)')
    plt.title('振幅箱线图')
    
    plt.subplot(1, 3, 3)
    if 'Risk_Level' in df.columns:
        sns.boxplot(data=df, x='Risk_Level', y='Max_Amplitude_mm', order=['Low', 'Medium', 'High'])
        plt.xlabel('风险等级')
        plt.ylabel('最大振幅 (mm)')
        plt.title('不同风险等级的振幅分布')
    
    plt.tight_layout()
    plt.show()

## 4. 快速实验

In [ ]:
# 导入我们的模块
try:
    from data_processing import BridgeVIVDataProcessor
    from train import BridgeVIVTrainer
    print("✅ 模块导入成功!")
except ImportError as e:
    print(f"❌ 模块导入失败: {e}")
    print("请确保在project目录下运行，或检查路径设置")

In [ ]:
# 数据处理
processor = BridgeVIVDataProcessor(str(data_path))

print("🔄 开始数据处理...")
processor.load_data()
print(f"✅ 数据加载完成: {processor.raw_data.shape}")

processor.clean_data()
print("✅ 数据清洗完成")

processor.feature_engineering()
print("✅ 特征工程完成")

X, targets, feature_names = processor.prepare_ml_data()
print(f"✅ ML数据准备完成: 特征数={len(feature_names)}, 样本数={X.shape[0]}")
print(f"📋 任务: {list(targets.keys())}")

In [ ]:
# 查看特征
print("🔍 生成的特征列表:")
for i, name in enumerate(feature_names, 1):
    print(f"  {i:2d}. {name}")

## 5. 训练简单模型

In [ ]:
# 快速训练
config_path = project_root / 'config' / 'config.yaml'
print(f"配置文件: {config_path}")

if config_path.exists():
    trainer = BridgeVIVTrainer(str(config_path))
    
    print("🚀 开始训练线性模型...")
    results = trainer.train_single_model('linear')
    
    print("✅ 训练完成!")
    print(f"📊 结果键: {list(results.keys())}")
    
    # 显示结果
    if 'linear' in results:
        linear_results = results['linear']
        print("\n📈 线性模型性能:")
        
        if 'metrics' in linear_results:
            for task, metrics in linear_results['metrics'].items():
                print(f"\n🎯 {task} 任务:")
                if 'test' in metrics:
                    for metric, value in metrics['test'].items():
                        print(f"  {metric}: {value:.4f}")
else:
    print("❌ 配置文件不存在")

## 6. 下一步

🎉 恭喜! 你已经成功运行了第一个桥梁VIV风险评估实验。

### 接下来你可以:

1. **运行完整实验**:
   ```bash
   cd ../
   python src/experiments.py --config config/config.yaml
   ```

2. **训练更多模型**:
   ```bash
   python src/train.py --config config/config.yaml --models linear random_forest xgboost
   ```

3. **超参数优化**:
   ```bash
   python src/hyperparam_search.py --config config/config.yaml
   ```

4. **生成预测**:
   ```bash
   python src/predict.py --comprehensive --input ../bridge_dataset_fixed.csv --output results/predictions.csv
   ```

### 📁 查看结果:
- **实验结果**: `experiments/` 文件夹
- **预测结果**: `results/` 文件夹  
- **实验报告**: `results/` 中的markdown文件

In [ ]:
print("🎯 快速开始教程完成!")
print("\n🚀 现在你可以:")
print("1. 在终端运行完整实验: python src/experiments.py --config config/config.yaml")
print("2. 查看生成的结果文件")
print("3. 尝试不同的模型和参数")
print("\n📖 更多信息请查看 README.md")